In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import math
import random
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

class HEMSEnvHardcoded(gym.Env):
    """
    The definitive HEMS environment with a hardcoded set of appliances
    based on the provided research paper.
    """
    metadata = {"render.modes": ["human"]}

    def __init__(self, config):
        super().__init__()
        # Load data
        self.solar_df = pd.read_csv(config["solar_path"], parse_dates=["time"])
        self.solar_df.columns = self.solar_df.columns.str.strip()
        
        # Config and hardware
        self.config = config
        self.solar_panel_area = config["solar_panel_area"]
        self.max_battery = float(config["max_battery_kwh"])
        self.appliances = config["appliances"]
        
        self.n_steps = len(self.solar_df)
        self.dt_hours = 1 / 60.0 # Data is per-minute
        self.steps_per_day = 24 * 60

        # Action space (2^N where N is number of schedulable appliances)
        self.action_space = spaces.Discrete(2**len(self.appliances))
        
        # Observation space
        num_obs_features = 4 + len(self.appliances)
        self.observation_space = spaces.Box(low=-np.inf, high=np.inf, shape=(num_obs_features,), dtype=np.float32)
        
        self.time_step = 0
        self.battery_soc = 0.0
        self._reset_appliance_states()

    def _reset_appliance_states(self):
        self.appliance_states = [{"name": app["name"], "is_running": False, "steps_remaining": 0, "task_completed_today": False} for app in self.appliances]

    def _compute_solar_output_kw(self, solar_row):
        ghi = pd.to_numeric(solar_row.get("ghi_pyr"), errors='coerce')
        humidity = pd.to_numeric(solar_row.get("relative_humidity"), errors='coerce')
        if pd.isna(ghi) or pd.isna(humidity): return 0.0
        panel_efficiency = 0.18
        weather_factor = max(0.0, 1.0 - (humidity / 100.0))
        pv_kw = (ghi * self.solar_panel_area * panel_efficiency * weather_factor) / 1000.0
        return max(pv_kw, 0.0)

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.time_step = 0
        self.battery_soc = 0.5 * self.max_battery
        self._reset_appliance_states()
        obs, info = self._get_obs()
        return obs, info

    def _get_obs(self):
        idx = self.time_step % self.n_steps
        solar_row = self.solar_df.iloc[idx]
        solar_kw = self._compute_solar_output_kw(solar_row)
        soc_norm = self.battery_soc / self.max_battery if self.max_battery > 0 else 0.0
        hour = (self.time_step // 60) % 24
        hour_rad = (2.0 * math.pi * hour) / 24.0
        appliance_statuses = [1.0 if not s["task_completed_today"] else 0.0 for s in self.appliance_states]
        obs_list = [solar_kw, soc_norm, math.sin(hour_rad), math.cos(hour_rad)] + appliance_statuses
        return np.array(obs_list, dtype=np.float32), {}

    def _get_grid_price_for_hour(self, hour):
        return 46.85 if 19 <= hour <= 23 else 40.53

    def step(self, action):
        idx = self.time_step % self.n_steps
        solar_row = self.solar_df.iloc[idx]
        hour = (self.time_step // 60) % 24

        if self.time_step > 0 and self.time_step % self.steps_per_day == 0:
            for state in self.appliance_states:
                state["task_completed_today"] = False

        action_commands = [int(bit) for bit in np.binary_repr(action, width=len(self.appliances))]
        appliance_load_kw = 0
        for i, start_command in enumerate(action_commands):
            state = self.appliance_states[i]
            if start_command == 1 and not state["is_running"] and not state["task_completed_today"]:
                state["is_running"] = True
                state["steps_remaining"] = self.appliances[i]["duration_steps"]
                state["task_completed_today"] = True
            if state["is_running"]:
                appliance_load_kw += self.appliances[i]["power_kw"]
                state["steps_remaining"] -= 1
                if state["steps_remaining"] == 0: state["is_running"] = False

        # --- Hardcoded Base Load Calculation ---
        base_load_kw = 0.6 # Refrigerator is always on
        if 6 <= hour < 8: # Light 1 is on from 6:00 to 8:00
            base_load_kw += 0.03
        if 18 <= hour < 24: # Light 2 is on from 18:00 to 24:00
            base_load_kw += 0.03
        
        total_load_kwh = (base_load_kw + appliance_load_kw) * self.dt_hours
        
        solar_kwh_available = self._compute_solar_output_kw(solar_row) * self.dt_hours
        
        grid_energy, battery_discharge = 0.0, 0.0
        remaining_load = total_load_kwh
        solar_supplied = min(solar_kwh_available, remaining_load)
        remaining_load -= solar_supplied
        if self.battery_soc > 0 and remaining_load > 0:
            discharge = min(self.battery_soc, remaining_load / 0.95)
            battery_discharge = discharge
            self.battery_soc -= discharge
            remaining_load -= discharge * 0.95
        if remaining_load > 0:
            grid_energy = remaining_load
        if solar_kwh_available > solar_supplied:
            charge = min((solar_kwh_available - solar_supplied) * 0.95, self.max_battery - self.battery_soc)
            self.battery_soc += charge

        grid_cost = grid_energy * self._get_grid_price_for_hour(hour)
        battery_deg_cost = battery_discharge * self.config.get("battery_deg_cost_per_kwh", 0.01)
        
        reward = -(grid_cost + battery_deg_cost)
        
        terminated = False
        missed_task_today = False
        if (self.time_step + 1) % self.steps_per_day == 0:
            if not all(s["task_completed_today"] for s in self.appliance_states):
                terminated = True
                missed_task_today = True
        
        if not terminated:
            terminated = self.time_step >= (self.n_steps - 1) 

        info = { "total_cost": grid_cost + battery_deg_cost, "task_missed": missed_task_today }
        
        self.time_step += 1
        obs, _ = self._get_obs()
        return obs, reward, terminated, False, info

if __name__ == "__main__":
    
    # --- Hardcoded Appliance "Jobs" based on the paper ---
    appliance_jobs = [
        {"name": "WaterHeater", "power_kw": 3.0, "duration_steps": 120}, # 2 hours
        {"name": "AC1", "power_kw": 1.5, "duration_steps": 180}, # 3 hours
        {"name": "AC2", "power_kw": 2.0, "duration_steps": 120}, # 2 hours
        {"name": "RiceCooker1", "power_kw": 1.0, "duration_steps": 30}, # 0.5 hours
        {"name": "MicrowaveOven1", "power_kw": 2.0, "duration_steps": 30}, # 0.5 hours
        {"name": "Dishwasher1", "power_kw": 1.0, "duration_steps": 30}, # 0.5 hours
        {"name": "WashingMachine", "power_kw": 0.3, "duration_steps": 30}, # 0.5 hours
    ]
    
    env_config = {
        "solar_path": "solar.csv", 
        "solar_panel_area": 27.4,  # 8 kW
        "max_battery_kwh": 10.0,   # 15 kWh battery
        "appliances": appliance_jobs
    }
    
    def make_env():
        return HEMSEnvHardcoded(env_config)

    print("\n--- Training HEMS Agent with Hardcoded Appliances ---")
    vec_env = DummyVecEnv([make_env for _ in range(4)])
    vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=False, clip_obs=10.0)
    model = PPO("MlpPolicy", vec_env, policy_kwargs=dict(net_arch=dict(pi=[256, 256], vf=[256, 256])), verbose=0)
    model.learn(total_timesteps=300000, progress_bar=True)
    
    print("\n--- Evaluating Agent ---")
    all_infos = []
    obs = vec_env.reset()
    done = False
    while not done:
        action, _ = model.predict(obs, deterministic=True)
        obs, _, terminated, info = vec_env.step(action)
        all_infos.append(info[0])
        done = terminated[0]

    results = pd.DataFrame(all_infos)
    
    total_cost = results['total_cost'].sum()
    missed_tasks = results['task_missed'].sum()
    num_days = len(results) / (24 * 60)

    print("\n\n==============================================")
    print("===   Hardcoded HEMS Scheduling Summary    ===")
    print("==============================================")
    print(f"Total Simulation Days: {num_days:.0f}")
    print(f"Total Annual Energy Cost: {total_cost:,.2f} PKR")
    print(f"Total Days with Missed Tasks: {missed_tasks:.0f}")


--- Training HEMS Agent with Hardcoded Appliances ---



--- Evaluating Agent ---


===   Hardcoded HEMS Scheduling Summary    ===
Total Simulation Days: 46
Total Annual Energy Cost: 41,735.62 PKR
Total Days with Missed Tasks: 1
